In [1]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection
from PIL import Image
import torch
from pathlib import Path
import json

processor = AutoImageProcessor.from_pretrained("nickmuchi/yolos-small-finetuned-license-plate-detection")
model = AutoModelForObjectDetection.from_pretrained("nickmuchi/yolos-small-finetuned-license-plate-detection")

image_dir = Path("./dataset/test/images")
output_file = Path("./yolos_predictions.json")
predictions = []

for img_path in image_dir.glob("*.jpg"):
    image = Image.open(img_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(outputs, target_sizes=target_sizes, threshold=0.5)[0]

    pred_list = []
    for box, score, label in zip(results["boxes"], results["scores"], results["labels"]):
        pred_list.append({
            "box": box.tolist(),
            "score": float(score),
            "label": int(label)
        })
    predictions.append({
        "image": img_path.name,
        "predictions": pred_list
    })
    print(f"Обработано: {img_path.name}, найдено {len(pred_list)} номеров")

with open(output_file, "w") as f:
    json.dump(predictions, f, indent=2)
print(f"Результаты сохранены в {output_file}")

C:\Users\lindo\Downloads\ALPR-RU\venv_yolos\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\lindo\Downloads\ALPR-RU\venv_yolos\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
C:\Users\lindo\Downloads\ALPR-RU\venv_yolos\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Could not find image processor class in the image processor config or the model config. Loading based on pattern matching with the model's feature extracto

Обработано: 09-19-06-Cam1_jpg.rf.a794b758a807ec6ac08b6d2f82e0f175.jpg, найдено 1 номеров
Обработано: 09-19-06-Cam2_jpg.rf.2110d0ac45022f2e9062bd320e03c1e1.jpg, найдено 1 номеров
Обработано: 10_jpg.rf.1d44824ffb1234f63f3d2531593524d3.jpg, найдено 1 номеров
Обработано: 11_6_2014_18_42_24_899_png.rf.8497e3135b77e92d28c28baea2923e5c.jpg, найдено 1 номеров
Обработано: 11_jpg.rf.90f13ef3216a7240666417476e8aca65.jpg, найдено 1 номеров
Обработано: 12_6_2014_19_54_57_849_png.rf.5f74991818bfb9bcb5b7ae9755b1b96d.jpg, найдено 2 номеров
Обработано: 15_jpg.rf.dcc03c3b96729ac3b36e69aa3053a197.jpg, найдено 1 номеров
Обработано: 20230405-112422_jpg.rf.69856ec911d503a58c6ad8db5c6a7537.jpg, найдено 0 номеров
Обработано: 22_jpg.rf.0d2db2ce3b52f80d5a111ddc51808342.jpg, найдено 1 номеров
Обработано: 34_jpg.rf.8c2bc72a29908b22a9cd0a26d2743c2c.jpg, найдено 1 номеров
Обработано: 52_jpg.rf.be9120d87e22941872dd6dac558016f7.jpg, найдено 1 номеров
Обработано: 81_jpg.rf.f229b6b499b877a6f420adf20d7ac9f1.jpg, найдено

In [1]:
import torch
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from PIL import Image
from pathlib import Path
import json

def read_yolo_labels(label_path, img_width, img_height):
    boxes = []
    labels = []
    if not label_path.exists():
        return boxes, labels
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 5:
                class_id = int(parts[0])
                x_c, y_c, w, h = map(float, parts[1:])
                x1 = (x_c - w/2) * img_width
                y1 = (y_c - h/2) * img_height
                x2 = (x_c + w/2) * img_width
                y2 = (y_c + h/2) * img_height
                boxes.append([x1, y1, x2, y2])
                labels.append(class_id)
    return boxes, labels

# Загрузка предсказаний YOLOS
with open("yolos_predictions.json", "r") as f:
    pred_data = json.load(f)

image_dir = Path("./dataset/test/images")
label_dir = Path("./dataset/test/labels")

preds = []
targets = []

for item in pred_data:
    img_name = item["image"]
    img_path = image_dir / img_name
    label_path = label_dir / img_name.replace(".jpg", ".txt").replace(".png", ".txt")

    img = Image.open(img_path)
    width, height = img.size

    # Предсказания YOLOS – label всегда 1, приведём к 0 (номерной знак)
    boxes = []
    scores = []
    labels_pred = []
    for pred in item["predictions"]:
        boxes.append(pred["box"])
        scores.append(pred["score"])
        labels_pred.append(0)  # преобразуем класс в 0

    if len(boxes) == 0:
        preds.append({
            "boxes": torch.zeros((0, 4), dtype=torch.float32),
            "scores": torch.tensor([], dtype=torch.float32),
            "labels": torch.tensor([], dtype=torch.int64)
        })
    else:
        preds.append({
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "scores": torch.tensor(scores, dtype=torch.float32),
            "labels": torch.tensor(labels_pred, dtype=torch.int64)
        })

    # Ground truth
    gt_boxes, gt_labels = read_yolo_labels(label_path, width, height)
    if len(gt_boxes) == 0:
        targets.append({
            "boxes": torch.zeros((0, 4), dtype=torch.float32),
            "labels": torch.tensor([], dtype=torch.int64)
        })
    else:
        targets.append({
            "boxes": torch.tensor(gt_boxes, dtype=torch.float32),
            "labels": torch.tensor(gt_labels, dtype=torch.int64)
        })

# Расчёт mAP
metric = MeanAveragePrecision()
metric.update(preds, targets)
result = metric.compute()

print(f"mAP50: {result['map_50'].item():.4f}")
print(f"mAP50-95: {result['map'].item():.4f}")
print(f"Precision (average per class): {result['map_per_class'].mean().item():.4f}")
print(f"Recall (mar_100): {result['mar_100'].item():.4f}")


C:\Users\lindo\Downloads\ALPR-RU\venv_yolos\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\lindo\Downloads\ALPR-RU\venv_yolos\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


mAP50: 0.8503
mAP50-95: 0.4952
Precision (average per class): -1.0000
Recall (mar_100): 0.6209
